## 🔹 **Cell 1 — Imports & Setup**

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torchvision.models import convnext_tiny
from torch.utils.data import Dataset, DataLoader
import os
from PIL import Image
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 🔹 **Cell 2 — Dataset & Intelligent Noise Filtering**

In [ ]:
import cv2

def is_blurry(img_np, threshold=50):
    gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
    variance = cv2.Laplacian(gray, cv2.CV_64F).var()
    return variance < threshold

class ElephantDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform
        self.class_to_idx = {}
        
        for idx, folder in enumerate(os.listdir(root_dir)):
            folder_path = os.path.join(root_dir, folder)
            if not os.path.isdir(folder_path):
                continue
            
            self.class_to_idx[folder] = idx
            
            for img_name in os.listdir(folder_path):
                img_path = os.path.join(folder_path, img_name)
                
                try:
                    with Image.open(img_path) as img:
                        img_np = np.array(img.convert("RGB"))
                    
                    # Size Check
                    h, w, _ = img_np.shape
                    if h < 100 or w < 50:
                        continue
                        
                    # Brightness Check
                    if img_np.mean() < 40:
                        continue
                        
                    # Contrast Check
                    if img_np.std() < 20:
                        continue
                        
                    # Blur Check
                    if is_blurry(img_np, threshold=50):
                        continue
                        
                except Exception:
                    continue  # skip unreadable images
                
                self.samples.append((img_path, idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        image = Image.open(path).convert("RGB")

        # Background Mitigation: Center Crop
        w, h = image.size
        image = image.crop((int(w*0.2), int(h*0.1), int(w*0.8), int(h*0.9)))

        if self.transform:
            image = self.transform(image)

        return image, label

## 🔹 **Cell 3 — Transforms & Loader**

In [ ]:
transform = transforms.Compose([
    transforms.Resize((256, 128)),
    transforms.ToTensor()
])

from pathlib import Path
DATASET_PATH = Path('/kaggle/input/datasets/girishcodes/restructured-elephant-dataset/restructured')

dataset = ElephantDataset(str(DATASET_PATH), transform)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

print("Total samples:", len(dataset))

## 🔹 **Cell 3.5 — Visual Inspection (FAST DIAGNOSTIC)**

In [ ]:
import matplotlib.pyplot as plt
import random

def show_samples(dataset, n=12):
    indices = random.sample(range(len(dataset)), min(n, len(dataset)))
    
    plt.figure(figsize=(12, 8))
    for i, idx in enumerate(indices):
        img, label = dataset[idx]
        img = img.permute(1,2,0).numpy()
        
        plt.subplot(3, 4, i+1)
        plt.imshow(img)
        plt.title(f"ID: {label}")
        plt.axis('off')
    
    plt.tight_layout()
    plt.show()

show_samples(dataset)

## 🔹 **Cell 4 — Simple Model (IMPORTANT CHANGE)**

In [ ]:
class SimpleReIDModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = convnext_tiny(weights="DEFAULT")
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(768, 256)

    def forward(self, x):
        feat = self.backbone.features(x)
        feat = self.pool(feat).flatten(1)
        emb = self.fc(feat)
        return F.normalize(emb, dim=1)

model = SimpleReIDModel().to(device)

## 🔹 **Cell 5 — Multi-Similarity Loss & Optimizer**

In [ ]:
class MultiSimilarityLoss(nn.Module):
    def __init__(self, alpha=2.0, beta=50.0, base=0.5):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.base = base

    def forward(self, embeddings, labels):
        sim = torch.matmul(embeddings, embeddings.t())

        labels = labels.unsqueeze(1)
        mask = labels == labels.t()

        loss = 0
        for i in range(len(embeddings)):
            pos = sim[i][mask[i]].clone()
            neg = sim[i][~mask[i]].clone()

            pos = pos[pos < 1]  # remove self
            
            # Semi-hard negative mining
            neg = neg[neg > 0.2]   # lower bound
            neg = neg[neg < 0.8]   # upper bound

            if len(pos) == 0 or len(neg) == 0:
                continue

            pos_loss = (1/self.alpha) * torch.log(
                1 + torch.sum(torch.exp(-self.alpha * (pos - self.base)))
            )

            neg_loss = (1/self.beta) * torch.log(
                1 + torch.sum(torch.exp(self.beta * (neg - self.base)))
            )

            loss += pos_loss + neg_loss

        return loss / len(embeddings)

criterion = MultiSimilarityLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

## 🔹 **Cell 6 — Training Loop (P×M Sampling)**

In [ ]:
from collections import defaultdict
import random

label_to_indices = defaultdict(list)

for idx, (_, label) in enumerate(dataset.samples):
    label_to_indices[label].append(idx)

EPOCHS = 5
P = 8
M = 4

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    steps = len(dataset) // (P * M)
    
    available_labels = list(label_to_indices.keys())
    if len(available_labels) < P:
        P = len(available_labels)
        steps = len(dataset) // (P * M) if len(dataset) // (P * M) > 0 else 1

    for _ in range(max(1, steps)):
        selected_labels = random.sample(available_labels, P)

        batch_indices = []
        for label in selected_labels:
            indices = label_to_indices[label]
            if len(indices) >= M:
                batch_indices.extend(random.sample(indices, M))
            else:
                batch_indices.extend(random.choices(indices, k=M))

        batch = [dataset[i] for i in batch_indices]

        imgs = torch.stack([x[0] for x in batch]).to(device)
        labels = torch.tensor([x[1] for x in batch]).to(device)

        embeddings = model(imgs)

        loss = criterion(embeddings, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

## 🔹 **Cell 7 — Compute Embeddings (VERY IMPORTANT)**

In [ ]:
model.eval()

all_embeddings = []
all_labels = []

with torch.no_grad():
    for imgs, labels in loader:
        imgs = imgs.to(device)
        emb = model(imgs)

        all_embeddings.append(emb.cpu())
        all_labels.append(labels)

all_embeddings = torch.cat(all_embeddings)
all_labels = torch.cat(all_labels)

print("Embeddings shape:", all_embeddings.shape)

## 🔹 **Cell 8 — Similarity Analysis (MOST IMPORTANT STEP)**

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

emb_np = all_embeddings.numpy()
labels_np = all_labels.numpy()

sim_matrix = cosine_similarity(emb_np)

same_sim = []
diff_sim = []

n = len(labels_np)

for i in range(n):
    for j in range(i+1, n):
        if labels_np[i] == labels_np[j]:
            same_sim.append(sim_matrix[i][j])
        else:
            diff_sim.append(sim_matrix[i][j])

print("Same Elephant Similarity:")
print("Mean:", np.mean(same_sim))
print("Min:", np.min(same_sim))
print("Max:", np.max(same_sim))

print("\nDifferent Elephant Similarity:")
print("Mean:", np.mean(diff_sim))
print("Min:", np.min(diff_sim))
print("Max:", np.max(diff_sim))

## 🔹 **Cell 9 — Similarity-Based Grouping (No Retraining)**

In [ ]:
THRESH_HIGH = 0.75
THRESH_LOW = 0.5

n = len(sim_matrix)
unassigned = set(range(n))

clusters = []
uncertain = []

while unassigned:
    i = unassigned.pop()
    group = [i]

    # Find strong neighbors from remaining unassigned
    candidates = []
    for j in unassigned:
        if sim_matrix[i][j] > THRESH_HIGH:
            candidates.append(j)

    # Validate EACH candidate against all current group members before adding
    for j in candidates:
        valid = True
        for g in group:
            if sim_matrix[j][g] < THRESH_LOW:
                valid = False
                break
        
        if valid:
            group.append(j)

    # Remove accepted members from unassigned pool
    for g in group:
        if g in unassigned:
            unassigned.remove(g)

    # Singletons go to uncertain — they had no confident match
    if len(group) == 1:
        uncertain.append(group[0])
    else:
        clusters.append(group)

print("Clusters formed:", len(clusters))
print("Uncertain samples:", len(uncertain))

# Show size of first 3 clusters
for idx, c in enumerate(clusters[:3]):
    print(f"Cluster {idx+1} size: {len(c)}")

## 🔹 **Cell 10 — Confidence-Classified Suggestion System**

In [ ]:
TOP_K = 5
HIGH_CONF = 0.70
MED_CONF  = 0.65
LOW_CONF  = 0.60
GAP_MIN   = 0.08  # top1 - top2 must be > this to accept as confident match

suggestions = {}

for i in range(n):
    sims = sim_matrix[i]
    
    top_indices = np.argsort(-sims)
    top_indices = [idx for idx in top_indices if idx != i][:TOP_K]

    classified = []
    for idx in top_indices:
        score = sims[idx]
        if score >= HIGH_CONF:
            level = "HIGH"
        elif score >= MED_CONF:
            level = "MEDIUM"
        elif score >= LOW_CONF:
            level = "LOW"
        else:
            continue  # skip garbage matches
        classified.append((idx, score, level))

    suggestions[i] = classified

# Show examples for first 5 images
for i in range(5):
    print(f"\nImage {i} suggestions:")
    if suggestions[i]:
        for idx, score, level in suggestions[i]:
            print(f"  -> Image {idx}, sim: {score:.3f}, confidence: {level}")
    else:
        print("  (no confident matches — likely new identity)")

# Apply tightened top1/top2 gap rule
print("\n--- GAP-VALIDATED MATCHES ---")
accepted = 0
ambiguous = 0
no_match = 0

for i in range(n):
    s = suggestions[i]
    if len(s) == 0:
        no_match += 1
    elif len(s) >= 2:
        top1_score = s[0][1]
        top2_score = s[1][1]
        gap = top1_score - top2_score
        if gap > 0.08 and top1_score >= 0.70:
            accepted += 1
        elif top1_score >= 0.60:
            ambiguous += 1
        else:
            no_match += 1
    else:
        # Only one candidate — accept if HIGH match strength
        if s[0][1] >= 0.70:
            accepted += 1
        else:
            ambiguous += 1

print(f"Accepted (STRONG MATCH): {accepted}")
print(f"Ambiguous (POSSIBLE/WEAK): {ambiguous}")
print(f"No match (NEW IDENTITY): {no_match}")

## 🔹 **Cell 11 — Save Model**

In [ ]:
torch.save(model.state_dict(), "/kaggle/working/ms_loss_hardmining_filtered_model_v2.pth")
print("Model saved!")